# 03b — Rectified Flow Training

Train a **Rectified Flow** model on the same data pipeline as `03_diffusion_training.ipynb`.
Shares the `DiffusionTransformer1D` backbone; only the generative process and training loop differ.

| Item | Value |
|------|-------|
| Process | Linear interpolation  x_t = (1−t)·x₀ + t·ε,  t ∈ [0,1] |
| Loss | MSE velocity + λ·freq  (λ=0.05) |
| Sampler | Euler ODE, 50 steps  (t: 1→0) |
| CFG | v_guided = (1+s)·v_cond − s·v_uncond |
| Backbone | DiffusionTransformer1D (identical to DDPM run) |

**Prerequisites**: run `02_clustering.ipynb` to produce `clusters.csv`.

In [ ]:

# ── Environment bootstrap (same as notebook 03) ──────────────────────────────
import importlib
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path


def clone_repo(repo_url, target_dir):
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', repo_url, str(target_dir)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'git clone failed')


def download_repo_archive(archive_url, runtime_dir, repo_dir, extract_dir, archive_path):
    if archive_path.exists():
        archive_path.unlink()
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    urllib.request.urlretrieve(archive_url, archive_path)
    with zipfile.ZipFile(archive_path, 'r') as archive:
        archive.extractall(runtime_dir)
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    shutil.move(str(extract_dir), str(repo_dir))


def find_or_bootstrap_repo_root():
    candidates = [
        Path.cwd().resolve(),
        Path('/tmp/vscode-colab/tesina'),
        Path('/content/tesina'),
        Path('/home/nicola/Desktop/Supsi/tesina'),
    ]
    for base in candidates:
        for candidate in [base, *base.parents]:
            if (candidate / 'src').exists() and (candidate / 'data').exists():
                return candidate

    runtime_dir  = Path(tempfile.gettempdir()) / 'vscode-colab'
    repo_dir     = runtime_dir / 'tesina'
    archive_path = runtime_dir / 'tesina.zip'
    extract_dir  = runtime_dir / 'tesina-master'
    repo_url     = 'https://github.com/ncapac/tesina.git'
    archive_url  = 'https://codeload.github.com/ncapac/tesina/zip/refs/heads/master'

    runtime_dir.mkdir(parents=True, exist_ok=True)
    if repo_dir.exists() and not (repo_dir / 'src').exists():
        shutil.rmtree(repo_dir)
    if not repo_dir.exists():
        try:
            clone_repo(repo_url, repo_dir)
        except Exception:
            download_repo_archive(archive_url, runtime_dir, repo_dir, extract_dir, archive_path)
    if (repo_dir / 'src').exists() and (repo_dir / 'data').exists():
        return repo_dir
    raise RuntimeError('Could not locate or bootstrap the tesina project root.')


REPO_ROOT      = find_or_bootstrap_repo_root()
DATA_DIR       = REPO_ROOT / 'data'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx

# Reload src modules so that edits to source files take effect without kernel restart
import src.data.loader, src.data.dataset
import src.models.transformer1d, src.models.rectified_flow
import src.training.train_rf
importlib.reload(src.data.loader)
importlib.reload(src.data.dataset)
importlib.reload(src.models.transformer1d)
importlib.reload(src.models.rectified_flow)
importlib.reload(src.training.train_rf)

from src.data.loader           import load_raw, compute_stats, normalize, denormalize
from src.data.dataset          import make_windows, train_val_split, numpy_dataloader
from src.models.transformer1d  import DiffusionTransformer1D
from src.models.rectified_flow import RectifiedFlowProcess
from src.training.train_rf     import RFTrainer

plt.rcParams['figure.dpi'] = 110
print('Project root:', REPO_ROOT)
print('JAX devices :', jax.devices())

# ── Training budget ──────────────────────────────────────────────────────────
QUICK_RUN  = True          # set False for ~200-epoch GPU run
N_EPOCHS   = 5   if QUICK_RUN else 200
BATCH_SIZE = 64
LR         = 1e-3
WARMUP_STEPS   = 300  if QUICK_RUN else 2000
GUIDANCE_SCALE = 1.5

CHECKPOINT_NAME = 'rf_best_model.pkl'


## 1. Load & prepare data

In [ ]:
df = load_raw(DATA_DIR / 'power.pk')

clusters_df    = pd.read_csv(DATA_DIR / 'clusters.csv')
cluster_labels = clusters_df['cluster_id'].values
N_CLUSTERS     = int(cluster_labels.max()) + 1

print(f'{df.shape[1]} meters, {N_CLUSTERS} clusters')
print(f'Cluster sizes: { {c: (cluster_labels==c).sum() for c in range(N_CLUSTERS)} }')

timestamps = df.index if isinstance(df.index, pd.DatetimeIndex) else None

stats    = compute_stats(df, cluster_labels)
df_norm  = normalize(df, stats, cluster_labels)

xs, cs, mid = make_windows(df_norm, cluster_labels, timestamps)
print(f'Windows: xs={xs.shape}  cs={cs.shape}')

x_train, c_train, x_val, c_val = train_val_split(xs, cs, mid, n_meters=df.shape[1])
print(f'Train: {x_train.shape[0]}  Val: {x_val.shape[0]}')

print('\nWindows per cluster (train):')
for cid in range(N_CLUSTERS):
    n_tr = (c_train[:, 0] == cid).sum()
    n_va = (c_val[:,   0] == cid).sum()
    print(f'  Cluster {cid}: train={n_tr:6d}  val={n_va:5d}')

print('\nNormalised stats per cluster (train):')
for cid in range(N_CLUSTERS):
    mask = c_train[:, 0] == cid
    vals = x_train[mask].ravel()
    print(f'  Cluster {cid}: mean={vals.mean():+.3f}  std={vals.std():.3f}')

## 2. Model & RF process setup

In [ ]:
key = jax.random.PRNGKey(42)   # different seed from DDPM to get independent init

model = DiffusionTransformer1D(
    seq_len     = 24,
    d_model     = 128,
    n_heads     = 4,
    n_layers    = 4,
    d_ff        = 256,
    n_clusters  = N_CLUSTERS,
    n_day_types = 2,
    n_months    = 12,
    n_dow       = 7,
    key         = key,
)

rf = RectifiedFlowProcess(freq_loss_weight=0.05)

total_steps = N_EPOCHS * max(1, len(x_train) // BATCH_SIZE)
trainer = RFTrainer(
    model, rf,
    lr            = LR,
    warmup_steps  = WARMUP_STEPS,
    total_steps   = total_steps,
    checkpoint_dir= str(CHECKPOINT_DIR),
)

n_params = sum(
    x.size for x in jax.tree_util.tree_leaves(eqx.filter(model, eqx.is_array))
)
print(f'Model params : {n_params:,}')
print(f'JAX device   : {jax.devices()[0]}')
print(f'Process      : Rectified Flow  (t ~ Uniform[0,1])')
print(f'Conditioning : c = [cluster_id, day_type, month, dow]  shape=(4,)')

# Load existing checkpoint if available (resume training)
rf_ckpt = CHECKPOINT_DIR / CHECKPOINT_NAME
if rf_ckpt.exists():
    trainer.load(CHECKPOINT_NAME)
    print(f'Resumed from checkpoint (step {trainer.step})')
else:
    print('No existing checkpoint — starting from scratch.')

## 3. Training

In [ ]:
train_loader = numpy_dataloader(
    x_train, c_train, batch_size=BATCH_SIZE, shuffle=True,  rng=np.random.default_rng(10)
)
val_loader = numpy_dataloader(
    x_val, c_val,   batch_size=BATCH_SIZE, shuffle=False, rng=np.random.default_rng(11)
)

trainer.fit(
    train_loader    = train_loader,
    val_loader      = val_loader,
    n_epochs        = N_EPOCHS,
    val_every       = 1,
    save_every      = max(1, N_EPOCHS // 5),
    log_every_steps = max(1, len(x_train) // BATCH_SIZE // 4),
    val_batches     = max(1, len(x_val) // BATCH_SIZE),
    log_cluster_losses = True,
)

train_losses = trainer.train_losses
val_losses   = trainer.val_losses
print(f'\nDone. Final train loss: {train_losses[-1]:.4f}  val loss: {val_losses[-1]:.4f}')

## 4. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(train_losses) + 1)
axes[0].plot(list(epochs), train_losses, 'o-', color='steelblue', lw=1.5, label='Train')
axes[0].plot(list(epochs), val_losses,   's-', color='coral',     lw=1.5, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Velocity-prediction loss')
axes[0].set_title('RF training and validation loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

if len(train_losses) > 1:
    rel = [(train_losses[0] - l) / train_losses[0] * 100 for l in train_losses]
    axes[1].plot(list(epochs), rel, 'o-', color='mediumseagreen', lw=1.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('% reduction from epoch-1 loss')
    axes[1].set_title('Relative loss improvement')
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(0, color='grey', ls='--', lw=0.8)

plt.tight_layout(); plt.show()
print(f'Epoch 1 loss : {train_losses[0]:.4f}')
print(f'Final loss   : {train_losses[-1]:.4f}  '
      f'({(train_losses[0]-train_losses[-1])/train_losses[0]*100:.1f}% reduction)')
if QUICK_RUN:
    print('\n⚠ QUICK_RUN=True: only 5 epochs. For thesis-quality results,')
    print('  set QUICK_RUN=False and run on GPU (~200 epochs).')

# ── Per-cluster loss curves ──────────────────────────────────────────────────
if trainer.cluster_losses:
    fig2, ax2 = plt.subplots(figsize=(9, 3.5))
    colors_cl = ['steelblue', 'coral', 'mediumseagreen']
    for cid, losses in sorted(trainer.cluster_losses.items()):
        ax2.plot(range(1, len(losses)+1), losses, 'o-',
                 color=colors_cl[cid % len(colors_cl)], lw=1.5, label=f'Cluster {cid}')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Eval loss')
    ax2.set_title('Per-cluster validation loss (RF)')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 5. Quick sample inspection (Euler 50 steps)

In [ ]:
hours  = np.arange(24)
n_show = 4

fig, axes = plt.subplots(N_CLUSTERS, 4, figsize=(16, 3.5 * N_CLUSTERS))
if N_CLUSTERS == 1:
    axes = axes[None, :]
day_labels = ['Weekday', 'Weekend']

for cid in range(N_CLUSTERS):
    for dt in range(2):
        rep_dow  = 1 if dt == 0 else 5
        c_batch  = jnp.array([[cid, dt, 5, rep_dow]] * n_show, dtype=jnp.int32)
        gen_key  = jax.random.PRNGKey(cid * 10 + dt + 100)
        samples_norm = rf.sample(
            trainer.model, c_batch,
            seq_len=24, batch_size=n_show,
            key=gen_key, n_steps=50, guidance_scale=GUIDANCE_SCALE,
        )
        samples_norm = np.array(samples_norm)

        real_mask   = (c_train[:, 0] == cid) & (c_train[:, 1] == dt)
        real_sample = x_train[real_mask][:n_show] if real_mask.sum() > 0 else None

        for k in range(n_show):
            ax = axes[cid, k]
            ax.plot(hours, samples_norm[k], color='coral',     lw=1.5, label='RF synth')
            if real_sample is not None and k < len(real_sample):
                ax.plot(hours, real_sample[k], color='steelblue', lw=1, ls='--',
                        alpha=0.7, label='real')
            ax.set_title(f'C{cid} {day_labels[dt]} #{k+1}', fontsize=8)
            ax.set_xlabel('Hour'); ax.tick_params(labelsize=7)
            if k == 0:
                ax.legend(fontsize=7)

plt.suptitle('RF sample inspection (Euler 50 steps, normalised scale)', fontsize=11)
plt.tight_layout(); plt.show()

## 6. Denormalised comparison

In [ ]:
hours = np.arange(24)
fig, axes = plt.subplots(N_CLUSTERS, 2, figsize=(13, 4 * N_CLUSTERS))
if N_CLUSTERS == 1:
    axes = axes[None, :]

for cid in range(N_CLUSTERS):
    for dt, day_name in enumerate(['Weekday', 'Weekend']):
        rep_dow = 1 if dt == 0 else 5
        c_batch = jnp.array([[cid, dt, 5, rep_dow]] * 100, dtype=jnp.int32)
        synth_norm = rf.sample(
            trainer.model, c_batch,
            seq_len=24, batch_size=100,
            key=jax.random.PRNGKey(cid * 100 + dt),
            n_steps=50, guidance_scale=GUIDANCE_SCALE,
        )
        synth_dn = denormalize(np.array(synth_norm), cid, stats)

        real_mask = (c_train[:, 0] == cid) & (c_train[:, 1] == dt)
        real_dn   = denormalize(x_train[real_mask], cid, stats)

        ax = axes[cid, dt]
        mu_r, s_r = real_dn.mean(0),  real_dn.std(0)
        mu_s, s_s = synth_dn.mean(0), synth_dn.std(0)
        ax.fill_between(hours, mu_r - s_r, mu_r + s_r, alpha=0.25, color='steelblue', label='Real ±σ')
        ax.fill_between(hours, mu_s - s_s, mu_s + s_s, alpha=0.25, color='coral',     label='RF ±σ')
        ax.plot(hours, mu_r, color='steelblue', lw=2)
        ax.plot(hours, mu_s, color='coral',     lw=2, ls='--')
        ax.set_title(f'Cluster {cid} — {day_name}')
        ax.set_xlabel('Hour'); ax.set_ylabel('Wh / hour')
        ax.legend(fontsize=7)

plt.suptitle('Real vs RF Synthetic daily profiles (Wh scale)', fontsize=12)
plt.tight_layout(); plt.show()

## 7. Save checkpoint

In [ ]:
trainer.save(CHECKPOINT_NAME)

n_params = sum(
    x.size for x in jax.tree_util.tree_leaves(eqx.filter(trainer.model, eqx.is_array))
)
print(f"  Parameters : {n_params:,}")
print(f"  Epochs done: {N_EPOCHS}  (QUICK_RUN={QUICK_RUN})")
if trainer.val_losses:
    print(f"  Best val   : {min(trainer.val_losses):.4f}  "
          f"(epoch {np.argmin(trainer.val_losses)+1})")

## 8. Observations

In [ ]:
print("=" * 60)
print("RF TRAINING RUN SUMMARY")
print("=" * 60)
print(f"\n  Process       : Rectified Flow  (t ~ Uniform[0,1])")
print(f"  Epochs        : {N_EPOCHS}  (QUICK_RUN={QUICK_RUN})")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Learning rate : {LR}  (cosine schedule, warmup={WARMUP_STEPS})")
print(f"  Steps/epoch   : {max(1, len(x_train) // BATCH_SIZE)}")
print(f"  Model params  : {sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(trainer.model, eqx.is_array))):,}")

if trainer.train_losses:
    first, last = trainer.train_losses[0], trainer.train_losses[-1]
    pct = (first - last) / first * 100
    print(f"\n  Initial loss  : {first:.4f}")
    print(f"  Final loss    : {last:.4f}  ({pct:.1f}% reduction)")
    if trainer.val_losses:
        best_ep  = int(np.argmin(trainer.val_losses)) + 1
        best_val = min(trainer.val_losses)
        print(f"  Best val loss : {best_val:.4f}  (epoch {best_ep})")

print(f"\n  Training device: {jax.devices()[0]}")
print(f"\n  Next step: run 05_comparison.ipynb after both DDPM and RF are fully trained.")
print("  Expected full-training metric targets (same as DDPM):")
print("    Discriminative accuracy <= 0.55")
print("    ACF L2 distance         <= 0.05")
print("    Wasserstein             <= 0.05")